In [2]:
import pickle
import numpy
import matplotlib.pyplot as plt
import pandas as pd

In [134]:
filename_fnirs = "013_FW_processed"
PID = filename_fnirs[0:3]
print(PID)
neural_df = pd.read_csv('013_FW_processed')
demo_data = open('data/human-watch-demo/013_FW_092024.pickle', 'rb')

demo_dict = dict(pickle.load(demo_data))
df = pd.DataFrame(demo_dict)

013


In [135]:
num = df["NumberOfDemos"]['seed']

try:
    new_dict = create_time_dict_robot(num, df)
except:
    new_dict = create_time_dict_game(num, df)
    

demo_df = pd.DataFrame(new_dict)
demo_df.head(4)


,floatTimestamps,dateTimestamps,actions,rewards,chosen_action_prob,optimal_action_prob,states
0,1.726864e+09,2024-09-20 20:24:42.473239990,NaN,NaN,None,None,"[1.0, 0.1953125, 0.390625, 1.0, 0.0, 1.0, 1.0,..."
1,1.726864e+09,2024-09-20 20:25:02.476663086,1.0,0.1,"[0.28802393977943536, 0.7119760602205647]","[0.28802393977943536, 0.7119760602205647]","[0.9861111111111112, 0.1953125, 0.390625, 1.0,..."
2,1.726864e+09,2024-09-20 20:25:02.478635010,1.0,0.1,"[0.25692738521349523, 0.7430726147865049]","[0.25692738521349523, 0.7430726147865049]","[0.9722222222222222, 0.1953125, 0.390625, 1.0,..."
3,1.726864e+09,2024-09-20 20:25:02.529635986,1.0,0.1,"[0.3132433061810999, 0.6867566938189]","[0.3132433061810999, 0.6867566938189]","[0.9583333333333334, 0.1953125, 0.390625, 1.0,..."


In [136]:
import datetime
from datetime import datetime, timedelta

def add_time(timestamp_str, diff):

    timestamp_obj = datetime.strptime(timestamp_str[:-3], "%Y-%m-%d %H:%M:%S.%f")
    new_timestamp_obj = timestamp_obj + timedelta(seconds=diff)
    new_timestamp_str = new_timestamp_obj.strftime("%Y-%m-%d %H:%M:%S.%f") #+ timestamp_obj[-3:]
    new_float_timestamp = new_timestamp_obj.timestamp()

    return new_timestamp_str, new_float_timestamp


In [137]:
first_time = neural_df["time"][0]
timestamp_obj = datetime.strptime(first_time[:-3], "%Y-%m-%d %H:%M:%S.%f")
diff = demo_df["floatTimestamps"][1] - timestamp_obj.timestamp()

neural_df[['dateTimestamps', 'floatTimestamps']] = neural_df['time'].apply(lambda x: pd.Series(add_time(x, diff)))

neural_df.head(4)

,Unnamed: 0,time,MARKER,pid,L_O_DSI,L_D_DSI,L_O_DSphi,L_D_DSphi,R_O_DSI,R_D_DSI,R_O_DSphi,R_D_DSphi,trial_id,dateTimestamps,floatTimestamps
0,0,2024-09-20 20:24:59.984092928,0,13,-0.027479,-0.003954,0.018600,0.034658,-1.733719,0.377201,8.011072,-1.490911,1,2024-09-20 16:25:02.476663,1.726864e+09
1,1,2024-09-20 20:25:00.926335232,0,13,0.010175,-0.002017,-0.110318,-0.000348,-1.425622,0.345761,8.339727,-1.348485,1,2024-09-20 16:25:03.418906,1.726864e+09
2,2,2024-09-20 20:25:00.996326912,0,13,0.046563,-0.000035,-0.241376,-0.032906,-1.125713,0.315393,8.673191,-1.212890,1,2024-09-20 16:25:03.488897,1.726864e+09
3,3,2024-09-20 20:25:01.057372928,0,13,0.080455,0.002011,-0.370741,-0.061969,-0.841723,0.286983,9.007031,-1.089117,1,2024-09-20 16:25:03.549943,1.726864e+09


In [138]:
demo_df['timestamps'] = pd.to_datetime(demo_df['dateTimestamps'])  # Convert to datetime
neural_df['timestamps'] = pd.to_datetime(neural_df['time'])  # Convert to datetime

merged_df = pd.merge_asof(
    demo_df.sort_values('timestamps'),  # Sort by timestamps before merging
    neural_df.sort_values('timestamps'),
    on='timestamps',                 # Merge on timestamps
    direction='nearest',             # Find the closest timestamp
    tolerance=pd.Timedelta(milliseconds=100)     # Tolerance for timestamp matching (adjust as needed)
)

merged_df.head(2)


/var/folders/bk/m5wzpcwd46xdzs4b0n4hp9ym0000gn/T/ipykernel_59963/1035702543.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  neural_df['timestamps'] = pd.to_datetime(neural_df['time'])  # Convert to datetime


,floatTimestamps_x,dateTimestamps_x,actions,rewards,chosen_action_prob,optimal_action_prob,states,timestamps,Unnamed: 0,time,...,L_D_DSI,L_O_DSphi,L_D_DSphi,R_O_DSI,R_D_DSI,R_O_DSphi,R_D_DSphi,trial_id,dateTimestamps_y,floatTimestamps_y
0,1.726864e+09,2024-09-20 20:24:42.473239990,NaN,NaN,None,None,"[1.0, 0.1953125, 0.390625, 1.0, 0.0, 1.0, 1.0,...",2024-09-20 20:24:42.473239990,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.726864e+09,2024-09-20 20:25:02.476663086,1.0,0.1,"[0.28802393977943536, 0.7119760602205647]","[0.28802393977943536, 0.7119760602205647]","[0.9861111111111112, 0.1953125, 0.390625, 1.0,...",2024-09-20 20:25:02.476663086,11.0,2024-09-20 20:25:02.470290944,...,0.022923,-0.856166,-0.181275,0.362455,0.191108,10.757741,-0.824827,1.0,2024-09-20 16:25:04.962861,1.726864e+09


In [139]:
# merged_df.fillna(method='ffill', inplace=True)

# merged_df.head(50)

merged_df = merged_df.dropna()
nans_per_column = merged_df.isna().sum()

print(nans_per_column)

floatTimestamps_x      0
dateTimestamps_x       0
actions                0
rewards                0
chosen_action_prob     0
optimal_action_prob    0
states                 0
timestamps             0
Unnamed: 0             0
time                   0
MARKER                 0
pid                    0
L_O_DSI                0
L_D_DSI                0
L_O_DSphi              0
L_D_DSphi              0
R_O_DSI                0
R_D_DSI                0
R_O_DSphi              0
R_D_DSphi              0
trial_id               0
dateTimestamps_y       0
floatTimestamps_y      0
dtype: int64


In [140]:
def create_time_dict_robot(num, demo_dict):
    new_dict = {'floatTimestamps':[],
                'dateTimestamps': [],
                'rewards':[],
                'chosen_actions': [],
                'optimal_actions': [],
                'states':[]}

    for i in range(num):
        for j in range(len(demo_dict[i]['rewards'])):
            new_dict['floatTimestamps'].append(demo_dict[i]["timestamps"][j])
            new_dict['dateTimestamps'].append(demo_dict[i]["timestamps_datetime"][j])
            new_dict['rewards'].append(demo_dict[i]["rewards"][j])
            new_dict['chosen_actions'].append(demo_dict[i]["chosen_actions"][j])
            new_dict['optimal_actions'].append(demo_dict[i]["optimal_actions"][j])
            new_dict['states'].append(demo_dict[i]["states"][j])

    return new_dict

def create_time_dict_game(num, demo_dict):
    new_dict = {'floatTimestamps':[],
                'dateTimestamps': [],
                'actions': [],
                'rewards':[],
                'chosen_action_prob': [],
                'optimal_action_prob': [],
                'states':[]}

    for i in range(num):
        for j in range(len(demo_dict[i]['rewards'])):
            new_dict['floatTimestamps'].append(demo_dict[i]["timestamps"][j])
            new_dict['dateTimestamps'].append(demo_dict[i]["timestamps_datetime"][j])
            new_dict['rewards'].append(demo_dict[i]["rewards"][j])
            new_dict['chosen_action_prob'].append(demo_dict[i]["chosen_action_prob"][j])
            new_dict['optimal_action_prob'].append(demo_dict[i]["optimal_action_prob"][j])
            new_dict['actions'].append(demo_dict[i]["actions"][j])
            new_dict['states'].append(demo_dict[i]["states"][j])

    return new_dict

In [141]:
import numpy as np

merged_df['binary_optimal'] = merged_df.apply(
    lambda row: 0 if np.array_equal(row['chosen_action_prob'], row['optimal_action_prob']) else 1, axis=1
)
merged_df.tail(2)

,floatTimestamps_x,dateTimestamps_x,actions,rewards,chosen_action_prob,optimal_action_prob,states,timestamps,Unnamed: 0,time,...,L_O_DSphi,L_D_DSphi,R_O_DSI,R_D_DSI,R_O_DSphi,R_D_DSphi,trial_id,dateTimestamps_y,floatTimestamps_y,binary_optimal
3457,1.726864e+09,2024-09-20 20:28:17.583434082,0.0,0.1,"[0.6767500855928905, 0.32324991440710954]","[0.8694381510536878, 0.1305618489463122]","[-0.04999999999999992, 0.2734375, 0.46875, 0.5...",2024-09-20 20:28:17.583434082,1029.0,2024-09-20 20:28:17.556998144,...,-6.23663,1.126699,4.320941,-0.551576,-8.7501,0.635025,1.0,2024-09-20 16:28:20.049569,1.726864e+09,1
3458,1.726864e+09,2024-09-20 20:28:17.633073975,0.0,0.1,"[0.4256730914964979, 0.5743269085035021]","[0.3502364498053999, 0.6497635501946001]","[-0.06388888888888881, 0.2734375, 0.46875, 0.5...",2024-09-20 20:28:17.633073975,1029.0,2024-09-20 20:28:17.556998144,...,-6.23663,1.126699,4.320941,-0.551576,-8.7501,0.635025,1.0,2024-09-20 16:28:20.049569,1.726864e+09,1


In [142]:
pd.DataFrame.to_csv(merged_df, "{}_FW".format(PID))